In [1]:
using DataFrames
using DataFramesMeta
using Query

┌ Info: Precompiling DataFramesMeta [1313f7d8-7da2-5740-9ea0-a2ca25f37964]
└ @ Base loading.jl:1260
┌ Info: Precompiling Query [1a8c2f83-1ff3-5112-b086-8aa67b057ba1]
└ @ Base loading.jl:1260


## DataFramesMeta

In [5]:
df = DataFrame(name=["John", "Sally", "Roger"],
               age=[54., 34., 79.],
               children=[0, 2, 4])

,name,age,children
,String,Float64,Int64
1,John,54.0,0
2,Sally,34.0,2
3,Roger,79.0,4


In [6]:
@linq df |>
    where(:age .> 40) |>
    select(n_children=:children, :name)

,n_children,name
,Int64,String
1,0,John
2,4,Roger


In [7]:
df = DataFrame(key=repeat(1:3, 4), value=1:12)

,key,value
,Int64,Int64
1,1,1
2,2,2
3,3,3
4,1,4
5,2,5
6,3,6
7,1,7
8,2,8
9,3,9


In [8]:
@linq df |>
    where(:value .> 3) |>
    by(:key, min=minimum(:value), max=maximum(:value)) |>
    select(:key, range=:max - :min)

,key,range
,Int64,Int64
1,1,6
2,2,6
3,3,6


In [9]:
@linq df |>
    groupby(:key) |>
    transform(value0=:value .- minimum(:value))

,key,value,value0
,Int64,Int64,Int64
1,1,1,0
2,1,4,3
3,1,7,6
4,1,10,9
5,2,2,0
6,2,5,3
7,2,8,6
8,2,11,9
9,3,3,0


## Query

In [10]:
df = DataFrame(name=["John", "Sally", "Roger"],
               age=[54., 34., 79.],
               children=[0, 2, 4])

,name,age,children
,String,Float64,Int64
1,John,54.0,0
2,Sally,34.0,2
3,Roger,79.0,4


In [11]:
q1 = @from i in df begin
    @where i.age > 40
    @select {n_children=i.children, i.name}
    @collect DataFrame
end

,n_children,name
,Int64,String
1,0,John
2,4,Roger


In [13]:
q2 = @from i in df begin
    @where i.age > 40
    @select {n_children=i.children, i.name}
end

n_children,name
0,"""John"""
4,"""Roger"""


In [14]:
total_children = 0
for i in q2
    global total_children += i.n_children
end
total_children

4

In [16]:
y = [i.name for i in q2 if i.n_children > 0]
y

1-element Array{String,1}:
 "Roger"

In [17]:
q3 = @from i in df begin
    @where i.age > 40 && i.children > 0
    @select i.name
    @collect
end

1-element Array{String,1}:
 "Roger"